# Figure for §7.2 — the occupancy scope condition

Produces the crossover chart, which is the single most important figure the
project has: it is where the recommendation stops being true.

| output | placeholder it fills |
|---|---|
| `occupancy-crossover.pdf` / `.png` | *the crossover chart, two panels* |
| `occupancy-crossover-patient-wait.csv` / `.md` | the left panel as numbers |
| `occupancy-crossover-impatient-served.csv` / `.md` | the right panel as numbers |

**Reads `results/runs` only** — no simulation, no CARLA server, seconds to
run, and safe while a campaign is writing to the same store.

## Why it has to be two panels

The claim under test is that demand-responsive routing pays while mean seat
occupancy stays under ~60% and stops paying above it. **What happens above
the line depends on whether passengers give up**, and a single panel can only
show one of the two answers:

* **Patient passengers** (nobody abandons): the ranking genuinely
  **inverts**. Reordering goes from beating fixed route to losing to it by
  thousands of seconds of wait.
* **Impatient passengers** (patience 900 s): the advantage **converges**
  toward zero without the best arm reliably crossing.

Showing only the left panel overstates the result; showing only the right
one hides that there is a regime where routing is actively worse. The two
together are the honest claim, and they are the reason §7.2 is written the
way it is.

Occupancy is not measured directly — it is Little's law,
`throughput × mean_ride_time`, as a percentage of capacity. The tables report
it so the reader can see which rungs were actually in the regime.

## Reading the left panel

Wait time is on a **log axis**. It has to be: the patient regime at the top
rung produces waits an order of magnitude larger than at the bottom, and on a
linear axis the interesting part is a flat line hugging zero. The shaded
region marks where the recommended policy is losing.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cycler import cycler
from scipy import stats

from src.experiment.store import DEFAULT_RUNS_DIR, load_frame

from IPython.display import Markdown, display

# Where the submission's figures are written. Outside the repo, beside the
# document that embeds them, so a figure and the prose quoting it move
# together. Override with AUTOBUS_FIGURES to render somewhere disposable.
FIGURES = Path(os.environ.get(
    "AUTOBUS_FIGURES",
    Path.home() / "Desktop" / "האוניברסיטה הפתוחה" / "סדנה"
    / "submission prep docs" / "figures"))
FIGURES.mkdir(parents=True, exist_ok=True)

# The campaign's palette, so every figure in the submission reads as one set.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#8b5cf6", "#d4a017"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "axes.spines.top": False,
    "axes.spines.right": False, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT, "xtick.labelsize": 9,
    "ytick.labelsize": 9, "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 110,
    # Text stays text in the PDF, so the document can be searched and the
    # figure scales without going fuzzy.
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save(figure, name: str) -> None:
    """
    Write one figure as both a vector PDF and a raster PNG.

    The PDF is what belongs in the submission document -- it stays sharp at
    any zoom and its labels remain selectable text. The PNG is for anything
    that cannot embed a PDF, such as slides or a video frame.

    :param figure: The matplotlib figure to write.
    :param name: Base filename, without an extension.
    :return: None.
    """
    figure.tight_layout()
    for suffix in ("pdf", "png"):
        path = FIGURES / f"{name}.{suffix}"
        figure.savefig(path, format=suffix, dpi=300, bbox_inches="tight")
    print(f"wrote {FIGURES / name}.pdf and .png")


def as_markdown(frame: pd.DataFrame) -> str:
    """
    Render a table as a Markdown block.

    Written out by hand rather than with ``DataFrame.to_markdown``, which
    needs the ``tabulate`` package -- not a dependency worth adding to the
    environment to format six rows.

    :param frame: The table to render.
    :return: A Markdown table, index included as the first column.
    """
    index_name = frame.index.name or ""
    header = [str(index_name)] + [str(c) for c in frame.columns]
    lines = ["| " + " | ".join(header) + " |",
             "|" + "|".join("---" for _ in header) + "|"]
    for key, row in frame.iterrows():
        cells = [f"{value:,.2f}" if isinstance(value, (int, float))
                 else str(value) for value in row]
        lines.append("| " + " | ".join([str(key)] + cells) + " |")
    return "\n".join(lines) + "\n"


def save_table(frame: pd.DataFrame, name: str) -> None:
    """
    Write one table as CSV and as a Markdown block ready to paste.

    :param frame: The table to write.
    :param name: Base filename, without an extension.
    :return: None.
    """
    frame.to_csv(FIGURES / f"{name}.csv")
    (FIGURES / f"{name}.md").write_text(as_markdown(frame), encoding="utf-8")
    print(f"wrote {FIGURES / name}.csv and .md")


def ci_half_width(values: pd.Series, confidence: float = 0.95) -> float:
    """
    Half-width of the confidence interval on a mean.

    The design review's SS3.2 bar is stated on this quantity, so it is
    computed the same way here as in ``summarize_comparison``.

    :param values: The per-seed observations.
    :param confidence: Confidence level.
    :return: Half-width in the units of ``values``, or nan if n < 2.
    """
    n = values.count()
    if n < 2:
        return float("nan")
    return float(stats.t.ppf(0.5 + confidence / 2, n - 1)
                 * values.std(ddof=1) / np.sqrt(n))


print("figures will be written to:", FIGURES)

## The sweep

In [ ]:
POLICIES = ["Fixed route", "Skip-stop", "Window W900", "Reordering W900",
            "Window W1800", "Reordering W1800"]


def crossover_runs(capacity: int, patient: bool) -> pd.DataFrame:
    """
    The load-crossover sweep at one capacity and one demand regime.

    ``max_wait_s`` is NaN for the patient arm, because "no patience" is
    stored as an absent value rather than a large one -- so the regime has
    to be selected with ``isna``, not with a comparison.

    :param capacity: Seats on the bus, 20 or 40.
    :param patient: True for the no-give-up regime.
    :return: One row per run.
    """
    frame = load_frame(DEFAULT_RUNS_DIR)
    chosen = frame[
        (frame["identity.world.map"] == "planar")
        & (frame["identity.world.stop_density"] == 0.5)
        & (frame["identity.world.map_seed"] == 42)
        & (frame["identity.trial.max_requests"] == 600)
        & (frame["identity.trial.capacity"] == capacity)
        & (frame["identity.policy"].isin(POLICIES))
    ]
    patience = chosen["identity.trial.max_wait_s"]
    chosen = chosen[patience.isna() if patient else patience.notna()]
    return chosen.assign(
        offered_per_min=60.0 / chosen["identity.trial.demand_interval_s"])


def curve(runs: pd.DataFrame, policy: str, column: str) -> pd.Series:
    """
    One policy's KPI against offered load.

    :param runs: Runs from ``crossover_runs``.
    :param policy: Policy label.
    :param column: KPI column.
    :return: Indexed by offered demand, sorted ascending.
    """
    block = runs[runs["identity.policy"] == policy]
    return block.groupby("offered_per_min")[column].mean().sort_index()


patient_20 = crossover_runs(20, patient=True)
patient_40 = crossover_runs(40, patient=True)
impatient_20 = crossover_runs(20, patient=False)
for name, block in (("patient cap 20", patient_20),
                    ("patient cap 40", patient_40),
                    ("impatient cap 20", impatient_20)):
    print(f"{name:<18} {len(block):>4} runs, "
          f"{block['offered_per_min'].nunique()} rungs, "
          f"{block['identity.seed'].nunique()} seeds")

## The figure

In [ ]:
# The two arms the panels compare: the project's recommendation against the
# baseline it is recommended over. Skip-stop joins the right panel because
# it, not fixed route, is what actually catches Reordering under load.
BASELINE, CONTENDER = "Fixed route", "Reordering W1800"

figure, axes = plt.subplots(1, 2, figsize=(11.6, 4.6))

# --- Left: patient wait, both capacities ---------------------------------
# Same policies, same map, same seeds; capacity is the only difference
# between solid and dashed. That is what makes this a crossover chart
# rather than two unrelated curves.
left = axes[0]
for policy, colour in ((BASELINE, SERIES[0]), (CONTENDER, SERIES[2])):
    for runs, capacity, style, alpha in ((patient_40, 40, "--", 0.55),
                                         (patient_20, 20, "-", 1.0)):
        series = curve(runs, policy, "kpi.mean_wait_time")
        left.plot(series.index, series.values, ls=style, lw=2.0,
                  marker="o" if capacity == 20 else None, markersize=5,
                  color=colour, alpha=alpha,
                  label=f"{policy}, {capacity} seats")
left.set_title("Patient passengers — the ranking inverts")
left.set_xlabel("offered demand (req/min)")
left.set_ylabel("mean wait time (s)")
left.set_yscale("log")
left.grid(axis="y")
left.set_axisbelow(True)
left.legend(fontsize=8, loc="upper left")

# Mark where the two solid lines swap order: the crossing is the finding,
# and eyeballing it off a log axis is exactly how it gets misread.
base_20 = curve(patient_20, BASELINE, "kpi.mean_wait_time")
cont_20 = curve(patient_20, CONTENDER, "kpi.mean_wait_time")
ahead = cont_20 < base_20
crossings = [r for previous, r in zip(ahead.index, ahead.index[1:])
             if ahead[previous] and not ahead[r]]
if crossings:
    at = crossings[0]
    left.axvspan(at, ahead.index.max(), color=SERIES[1], alpha=0.07)
    left.annotate("routing now loses",
                  xy=(at, cont_20[at]), xytext=(-6, 26),
                  textcoords="offset points", fontsize=9, color=SERIES[1],
                  ha="right",
                  arrowprops=dict(arrowstyle="->", color=SERIES[1], lw=1.2))

# --- Right: impatient coverage, capacity 20 ------------------------------
right = axes[1]
for policy, colour in ((BASELINE, SERIES[0]), ("Skip-stop", SERIES[1]),
                       (CONTENDER, SERIES[2])):
    series = curve(impatient_20, policy, "kpi.served_fraction") * 100
    right.plot(series.index, series.values, marker="o", markersize=5,
               lw=2.0, color=colour, label=policy)
right.set_title("Impatient passengers — it converges instead")
right.set_xlabel("offered demand (req/min)")
right.set_ylabel("served fraction (%)")
right.set_ylim(0, 100)
right.grid(axis="y")
right.set_axisbelow(True)
right.legend(fontsize=8, loc="lower left")

gap = (curve(impatient_20, CONTENDER, "kpi.served_fraction")
       - curve(impatient_20, BASELINE, "kpi.served_fraction")) * 100
# Clear of every curve: above the leading one on the left, below the
# converged bundle on the right.
right.annotate(f"gap {gap.iloc[0]:+.0f} pts", xy=(gap.index[0], 94),
               fontsize=9, color=INK_SOFT, ha="left")
right.annotate(f"{gap.iloc[-1]:+.0f} pts", xy=(gap.index[-1], 46),
               fontsize=9, color=INK_SOFT, ha="right")

for axis in axes:
    axis.set_xticks(sorted(impatient_20["offered_per_min"].unique()))
figure.suptitle("Campus map, capacity 20 unless marked, 30 seeds a point — "
                "what happens above ~60% seat occupancy",
                fontsize=10, color=INK_SOFT, y=1.02)
save(figure, "occupancy-crossover")
plt.show()

## The numbers behind it

Both panels as tables, with the Little's-law occupancy row underneath. The
two assertions at the end are the claims §7.2 makes; if the store ever stops
supporting them, this cell fails rather than the document quietly going
stale.

In [ ]:
def regime_table(runs: pd.DataFrame, column: str, scale: float = 1.0,
                 capacity: int = 20) -> pd.DataFrame:
    """
    One KPI for every policy against offered load, plus occupancy.

    :param runs: Runs from ``crossover_runs``.
    :param column: KPI column.
    :param scale: Multiplier, 100 for fractions.
    :param capacity: Seats, for the Little's-law occupancy row.
    :return: Policies as rows, offered demand as columns.
    """
    table = runs.pivot_table(index="identity.policy",
                             columns="offered_per_min",
                             values=column).reindex(POLICIES) * scale
    # Little's law: mean aboard = arrival rate x time aboard. Reported for
    # the busiest arm at each rung, because the scope condition is about
    # how full the bus actually got, not about an average over policies.
    seats = (runs.groupby(["identity.policy", "offered_per_min"])
             .apply(lambda g: 100.0 * g["kpi.throughput"].mean() / 60.0
                    * g["kpi.mean_ride_time"].mean() / capacity,
                    include_groups=False)
             .groupby("offered_per_min").max())
    table.loc["— seats used (%) —"] = seats
    table.columns = [f"{c:.2f}/min" for c in table.columns]
    return table


patient_wait = regime_table(patient_20, "kpi.mean_wait_time")
impatient_served = regime_table(impatient_20, "kpi.served_fraction", 100.0)

display(Markdown("**Patient regime, capacity 20 — mean wait (s).** "
                 "The bottom row is why the ranking moves."))
display(patient_wait.round(1))
save_table(patient_wait.round(1), "occupancy-crossover-patient-wait")

display(Markdown("**Impatient regime, capacity 20 — served fraction (%).**"))
display(impatient_served.round(1))
save_table(impatient_served.round(1),
           "occupancy-crossover-impatient-served")

# The two claims section 7.2 makes, checked against what was just plotted.
top = patient_wait.columns[-1]
patient_inverts = (patient_wait.loc["Reordering W1800", top]
                   > patient_wait.loc["Fixed route", top])
best_impatient = impatient_served.drop(index="— seats used (%) —")[top].max()
impatient_holds = best_impatient >= impatient_served.loc["Fixed route", top]
print(f"at {top}, patient: Reordering W1800 "
      f"{patient_wait.loc['Reordering W1800', top]:,.0f} s vs fixed route "
      f"{patient_wait.loc['Fixed route', top]:,.0f} s -> "
      f"{'INVERTS' if patient_inverts else 'no inversion'}")
print(f"at {top}, impatient: best arm {best_impatient:.1f}% vs fixed route "
      f"{impatient_served.loc['Fixed route', top]:.1f}% -> "
      f"{'no crossing' if impatient_holds else 'CROSSES'}")
print(f"peak occupancy reached: "
      f"{patient_wait.loc['— seats used (%) —'].max():.0f}% of 20 seats")
assert patient_inverts, "the patient inversion no longer reproduces"
assert impatient_holds, "the impatient best arm now falls below fixed route"